# Attention Surgery

Move beyond observing attention patterns to surgically manipulating them.
This notebook demonstrates attention knockout, pattern patching, and
per-edge attribution — the tools for understanding *which specific
attention edges* drive model behavior.

This notebook covers the `irtk.attention_surgery` module.

## Why This Matters

Attention surgery lets you directly modify attention patterns — knocking out specific heads, forcing attention to particular positions, or patching patterns between runs. These interventions establish causal claims about what attention heads actually do.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import attention_surgery

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Attention Knockout

What happens when we prevent a head from attending to a specific position?
Knockout sets the pre-softmax score to -inf and lets softmax renormalize.

In [ ]:
prompt = "The Eiffel Tower is in"
tokens = model.to_tokens(prompt)
token_strs = [tokenizer.decode([int(t)]) for t in tokens]
print(f"Tokens: {token_strs}")

paris_id = tokenizer.encode(" Paris")[0]

# Clean prediction
clean_logits = model(tokens)
clean_prob = float(jax.nn.softmax(clean_logits[-1])[paris_id])
print(f"\nP(' Paris') clean: {clean_prob:.4f}")

# Knock out attention from last position to each key position at layer 9
for k_pos in range(len(tokens)):
    logits = attention_surgery.attention_knockout(
        model, tokens, layer=9, head=9, query_pos=-1, key_pos=k_pos
    )
    prob = float(jax.nn.softmax(logits[-1])[paris_id])
    delta = prob - clean_prob
    print(f"  KO key={token_strs[k_pos]:>12s}: P(' Paris')={prob:.4f} (Δ={delta:+.4f})")

## 2. Attention Knockout Matrix

Knock out multiple attention edges at once using a boolean mask.

In [ ]:
# Block the last position from attending to tokens 1-3 ("Eiffel Tower")
seq_len = len(tokens)
mask = np.zeros((seq_len, seq_len), dtype=bool)
mask[-1, 1:4] = True  # block last pos from attending to positions 1-3

print(f"Masking last position from attending to: {[token_strs[i] for i in range(1, 4)]}")

logits = attention_surgery.attention_knockout_matrix(
    model, tokens, layer=9, head=9, mask=mask
)
prob = float(jax.nn.softmax(logits[-1])[paris_id])
print(f"P(' Paris') with subject knockout: {prob:.4f} (clean: {clean_prob:.4f})")

## 3. Attention Pattern Patching

Patch the attention pattern from a different run. This isolates the
effect of *where* a head attends, holding everything else constant.

In [ ]:
clean_prompt = "The Eiffel Tower is in"
corrupted_prompt = "The Big Ben is located in"

clean_tokens = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)

print(f"Clean: {clean_prompt!r}")
print(f"Corrupted: {corrupted_prompt!r}")

# Patch attention pattern from corrupted into clean at different layers
for layer in [0, 3, 6, 9, 11]:
    logits = attention_surgery.attention_pattern_patch(
        model, clean_tokens, corrupted_tokens, layer=layer, head=0
    )
    prob = float(jax.nn.softmax(logits[-1])[paris_id])
    print(f"  L{layer}H0 pattern patched: P(' Paris')={prob:.4f}")

## 4. Force Attention

Force a head to use a specific attention pattern — e.g., uniform
attention, or attending only to a specific token.

In [ ]:
seq_len = len(tokens)

# Uniform attention (causal)
uniform = np.tril(np.ones((seq_len, seq_len)))
uniform = uniform / uniform.sum(axis=-1, keepdims=True)

# Attend only to BOS
bos_only = np.zeros((seq_len, seq_len))
bos_only[:, 0] = 1.0

# Self-attention only
self_only = np.eye(seq_len)

patterns = [
    ("Uniform", uniform),
    ("BOS only", bos_only),
    ("Self only", self_only),
]

for name, pattern in patterns:
    logits = attention_surgery.force_attention(
        model, tokens, layer=9, head=9, target_pattern=pattern
    )
    prob = float(jax.nn.softmax(logits[-1])[paris_id])
    top = int(jnp.argmax(logits[-1]))
    print(f"  {name:12s}: P(' Paris')={prob:.4f}  top={tokenizer.decode([top])!r}")

## 5. Attention Edge Attribution

Which (query, key) edges are most important for the model's prediction?
This systematically knocks out each edge and measures the metric change.

In [ ]:
# Short prompt for tractable edge attribution
prompt = "Paris is in"
tokens = model.to_tokens(prompt)
token_strs = [tokenizer.decode([int(t)]) for t in tokens]

france_id = tokenizer.encode(" France")[0]
def metric(logits):
    return float(logits[-1, france_id])

# Compute edge attribution for a few heads
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (layer, head) in zip(axes, [(0, 0), (6, 9), (11, 10)]):
    attr = attention_surgery.attention_edge_attribution(
        model, tokens, layer=layer, head=head, metric_fn=metric
    )
    im = ax.imshow(attr, cmap="Reds", aspect="auto")
    ax.set_xticks(range(len(token_strs)))
    ax.set_xticklabels(token_strs, rotation=45, ha="right")
    ax.set_yticks(range(len(token_strs)))
    ax.set_yticklabels(token_strs)
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    ax.set_title(f"L{layer}H{head}")
    plt.colorbar(im, ax=ax)

plt.suptitle("Attention Edge Attribution for ' France' logit", fontsize=14)
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `attention_knockout()` | Zero a specific (query, key) attention edge |
| `attention_knockout_matrix()` | Knockout multiple edges with a boolean mask |
| `attention_pattern_patch()` | Patch attention pattern from another run |
| `force_attention()` | Force a head to use a specific pattern |
| `attention_edge_attribution()` | Per-edge attribution matrix |